# Phase 3 proper — the winning recipe × 3 seeds + restraint probe

**Runtime: L4.** ~40 min, ~4 units.

The A/B picked **no-trace × 1 epoch**. Seed 3407 already exists (notebook 04's
`sft_notrace_s3407_ep1` on Drive), so this notebook:
1. Trains the SAME recipe with seeds **42** and **1234** (fresh model, fresh
   mixture shuffle per seed — honest pipeline variance, not just init variance)
2. Dev-evals each (61 bugs × k=16 — same protocol as 04)
3. Runs a **restraint probe** on all three seeds (incl. reloading 3407's adapter):
   show the model 32 CORRECT dev functions with the same repair prompt — does it
   leave them alone? We measure how often its output still passes the tests, and
   how often it returns the code literally unchanged.
4. Prints the cross-seed table: mean ± spread. **That spread is our real noise
   ruler** (the base2 trick in 04c measured determinism, not noise — same seed,
   same RNG stream, identical samples).

After this: ONE held-out milestone eval of the SFT arm, then DPO.

In [ ]:
# Setup: Drive, fresh repo, data, routing
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, random, importlib, gc
PHASE2 = '/content/drive/MyDrive/rl-post-training/phase2'
PHASE3 = '/content/drive/MyDrive/rl-post-training/phase3'
!rm -rf /content/ptl
!git clone -q https://github.com/nidhi1603/post-training-lab.git /content/ptl
sys.path.insert(0, '/content/ptl/src')
for mod in ('prompts',):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])
from prompts import build_repair_prompt, extract_code
ver = !git -C /content/ptl log --oneline -1
print('FACTORY VERSION:', ver[0])
bugs = json.load(open('/content/ptl/data/data_v0_bugs.json'))
restraint = json.load(open('/content/ptl/data/data_v0_restraint.json'))
routing = {r['id']: r['pile'] for r in json.load(open(f'{PHASE2}/routing_v0.json'))}
train_bugs = [b for b in bugs if b['split'] == 'train']
dev_bugs = [b for b in bugs if b['split'] == 'dev']
dev_clean = [r for r in restraint if r['split'] == 'dev']
print(len(train_bugs), 'train |', len(dev_bugs), 'dev bugs |', len(dev_clean), 'dev clean (probe)')

In [ ]:
%%capture
!pip install unsloth

In [ ]:
# Shared helpers: grading, dev eval, restraint probe (model passed in explicitly)
import re, subprocess, tempfile, torch
from concurrent.futures import ThreadPoolExecutor
from unsloth import FastLanguageModel

def passes(code, rec, timeout=6):
    prog = '\n'.join(list(rec['test_imports']) + [code] + list(rec['test_list']))
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(prog); path = f.name
    try:
        return subprocess.run([sys.executable, path], capture_output=True, timeout=timeout).returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.unlink(path)

@torch.no_grad()
def generate_k(model, tok, source_code, test_list, k, batch=2):
    text = tok.apply_chat_template(
        [{'role': 'user', 'content': build_repair_prompt(source_code, test_list)}],
        tokenize=False, add_generation_prompt=True)
    enc = tok([text], return_tensors='pt', padding=True, padding_side='left').to('cuda')
    out = model.generate(**enc, do_sample=True, temperature=1.0, top_p=0.95,
                         num_return_sequences=k, max_new_tokens=512,
                         pad_token_id=tok.eos_token_id)
    gen = tok.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    return [extract_code(t) for t in gen]

def dev_eval(model, tok, tag, k=16):
    FastLanguageModel.for_inference(model)
    per_bug = []
    for b in dev_bugs:
        codes = generate_k(model, tok, b['buggy'], b['test_list'], k)
        with ThreadPoolExecutor(max_workers=4) as pool:
            flags = list(pool.map(lambda c: passes(c, b), codes))
        per_bug.append({'id': b['id'], 'n': k, 'c': sum(flags)})
    p1 = sum(r['c']/r['n'] for r in per_bug) / len(per_bug)
    p16 = sum(1 for r in per_bug if r['c'] > 0) / len(per_bug)
    res = {'tag': tag, 'pass@1': p1, 'pass@16': p16, 'gap': p16 - p1, 'per_bug': per_bug}
    json.dump(res, open(f'{PHASE3}/dev_eval_{tag}.json', 'w'))
    print(f"[{tag}]  pass@1={p1*100:.1f}%  pass@16={p16*100:.1f}%  gap={(p16-p1)*100:.1f}")
    return res

def restraint_probe(model, tok, tag, k=4):
    FastLanguageModel.for_inference(model)
    norm = lambda s: re.sub(r'\s+', ' ', s.strip())
    still_pass = unchanged = total = 0
    for r in dev_clean:
        codes = generate_k(model, tok, r['code'], r['test_list'], k)
        with ThreadPoolExecutor(max_workers=4) as pool:
            flags = list(pool.map(lambda c: passes(c, r), codes))
        still_pass += sum(flags)
        unchanged += sum(norm(c) == norm(r['code']) for c in codes)
        total += k
    res = {'tag': tag, 'still_pass': still_pass/total, 'unchanged': unchanged/total,
           'n_functions': len(dev_clean), 'k': k}
    json.dump(res, open(f'{PHASE3}/restraint_probe_{tag}.json', 'w'))
    print(f"[probe {tag}]  output still passes: {still_pass/total*100:.1f}%   returned unchanged: {unchanged/total*100:.1f}%")
    return res

In [ ]:
# The winning recipe as a function of seed (identical to notebook 04's cells)
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

def build_mixture(seed):
    rng = random.Random(seed)
    def bug_example(b):
        return {'prompt': build_repair_prompt(b['buggy'], b['test_list']),
                'answer': '```python\n' + b['fixed'].strip() + '\n```'}
    mixture = []
    for b in train_bugs:
        copies = 3 if routing.get(b['id']) == 'sft' else 1
        mixture += [bug_example(b)] * copies
    clean_train = [r for r in restraint if r['split'] == 'train']
    rng.shuffle(clean_train)
    for r in clean_train[:150]:
        mixture.append({'prompt': build_repair_prompt(r['code'], r['test_list']),
                        'answer': '```python\n' + r['code'].strip() + '\n```'})
    rng.shuffle(mixture)
    return mixture

def run_seed(seed):
    print(f'===== SEED {seed} =====')
    model, tok = FastLanguageModel.from_pretrained(
        'unsloth/Qwen2.5-Coder-1.5B-Instruct', max_seq_length=1024,
        load_in_4bit=True, dtype=None)
    model = FastLanguageModel.get_peft_model(
        model, r=32, lora_alpha=64, lora_dropout=0,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        use_gradient_checkpointing='unsloth', random_state=seed)
    def to_text(ex):
        msgs = [{'role': 'user', 'content': ex['prompt']},
                {'role': 'assistant', 'content': ex['answer']}]
        return {'text': tok.apply_chat_template(msgs, tokenize=False)}
    ds = Dataset.from_list(build_mixture(seed)).map(to_text)
    FastLanguageModel.for_training(model)
    trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds,
        dataset_text_field='text',
        args=SFTConfig(per_device_train_batch_size=8, gradient_accumulation_steps=2,
            num_train_epochs=1, learning_rate=2e-4, lr_scheduler_type='cosine',
            warmup_ratio=0.05, logging_steps=10, seed=seed, output_dir='/content/out',
            report_to='none', save_strategy='no'))
    trainer = train_on_responses_only(trainer,
        instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
    trainer.train()
    model.save_pretrained(f'{PHASE3}/sft_notrace_s{seed}_ep1')
    dev_eval(model, tok, f'ep1_seed{seed}')
    restraint_probe(model, tok, f'sft_s{seed}')
    del trainer, model, tok
    gc.collect(); torch.cuda.empty_cache()

for s in (42, 1234):
    run_seed(s)

In [ ]:
# Probe seed 3407 too (reload notebook 04's saved adapter from Drive)
model, tok = FastLanguageModel.from_pretrained(
    f'{PHASE3}/sft_notrace_s3407_ep1', max_seq_length=1024,
    load_in_4bit=True, dtype=None)
restraint_probe(model, tok, 'sft_s3407')
del model, tok
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# CROSS-SEED TABLE — the SFT arm's dev result, with its real noise ruler
SEEDS = (3407, 42, 1234)
rows = [json.load(open(f'{PHASE3}/dev_eval_ep1_seed{s}.json')) for s in SEEDS]
base = json.load(open(f'{PHASE3}/dev_eval_base_seed3407.json'))
print(f"{'tag':<14} pass@1   pass@16   gap")
print(f"{'base':<14} {base['pass@1']*100:6.1f}%  {base['pass@16']*100:6.1f}%  {base['gap']*100:5.1f}")
for r in rows:
    print(f"{r['tag']:<14} {r['pass@1']*100:6.1f}%  {r['pass@16']*100:6.1f}%  {r['gap']*100:5.1f}")
import statistics as st
for m in ('pass@1', 'pass@16'):
    vals = [r[m]*100 for r in rows]
    print(f"\n{m}: mean {st.mean(vals):.1f}%  spread (max-min) {max(vals)-min(vals):.1f} pts  sd {st.stdev(vals):.1f}")
print('\nNOISE RULER: the cross-seed spread above is the honest tie threshold for')
print('every future arm-vs-arm comparison on the dev slice.')
print('\nRestraint probes (does it leave correct code alone?):')
for s in SEEDS:
    p = json.load(open(f'{PHASE3}/restraint_probe_sft_s{s}.json'))
    print(f"  seed {s}: still-passes {p['still_pass']*100:.1f}%  unchanged {p['unchanged']*100:.1f}%")
print('\nGate: every seed must beat base pass@1 (45.9%). Bring the whole printout back.')

## Bring back to the session
1. The **cross-seed table** with mean, spread, and sd
2. The **restraint probe lines** (still-passes should stay high — a model that
   "fixes" correct code into broken code is over-eager, and that costs points on
   the real benchmark where some inputs need only small edits)
3. Training-loss curves if any seed looked unlike the others

After this: notebook 06 — merge the chosen SFT adapter and run the ONE held-out
milestone eval (frozen protocol, bigcode harness), then Phase 4 DPO.